# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip install duckdb --quiet

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
!pip install duckdb --quiet

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

march_agg = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS march_impressions,
        SUM(f.gsc_clicks) AS march_clicks,
        AVG(f.gsc_avg_position) AS march_avg_position,
        MAX(CASE
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) < 0 THEN NULL
            ELSE DATE_DIFF('day', c.content_updated_date, f.report_date)
        END) AS days_since_update
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# same client-grouped split as w05, same random_state -- reproduces the exact same train/test clients
unique_clients = march_agg['client_hash_id'].unique()
train_clients, test_clients = train_test_split(unique_clients, test_size=0.3, random_state=42)

labeled = march_agg.merge(april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
labeled['april_clicks'] = labeled['april_clicks'].fillna(0)
labeled['declining'] = (labeled['april_clicks'] < labeled['march_clicks'] * 0.8).astype(int)

train_df = labeled[labeled['client_hash_id'].isin(train_clients)].copy()
test_df = labeled[labeled['client_hash_id'].isin(test_clients)].copy()

features = ['march_impressions', 'march_clicks', 'march_avg_position', 'days_since_update']
train_df['days_since_update'] = train_df['days_since_update'].fillna(9999)
test_df['days_since_update'] = test_df['days_since_update'].fillna(9999)

X_train, y_train = train_df[features], train_df['declining']
X_test, y_test = test_df[features], test_df['declining']

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

print(f"Train pages: {len(train_df)}, Test pages: {len(test_df)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train pages: 102353, Test pages: 74385


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Section builds a ranked to do list ranked by" expected_clicks_at_risk", taking 15 clients for test runs. Each page gets a score so that pages with real traffic on the line rise to the top, not just pages the model happens to be confident about. one client dominates the top of the queue, but that's mostly because it's 42% of the test set, and a size comparison against another big client with a low decline rate showed the ranking isn't just echoing client size. Each page also gets an action label — Monitor, Refresh, or Deprioritize — based on whether it ranks well and how much visibility it has, plus a plain-English reason sentence explaining why. Also found that the model's probability score maxes out at 1.0 for very high-traffic pages, which means the "risk-weighting" mostly just becomes click count for the very biggest pages

In [ ]:
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

march_agg = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS march_impressions,
        SUM(f.gsc_clicks) AS march_clicks,
        AVG(f.gsc_avg_position) AS march_avg_position,
        MAX(CASE
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) < 0 THEN NULL
            ELSE DATE_DIFF('day', c.content_updated_date, f.report_date)
        END) AS days_since_update
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

labeled = march_agg.merge(april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
labeled['april_clicks'] = labeled['april_clicks'].fillna(0)
labeled['declining'] = (labeled['april_clicks'] < labeled['march_clicks'] * 0.8).astype(int)
labeled['days_since_update'] = labeled['days_since_update'].fillna(9999)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
known_staleness = labeled[labeled['days_since_update'] != 9999]
stale_threshold = known_staleness['days_since_update'].quantile(0.75)
good_position_threshold = labeled['march_avg_position'].quantile(0.25)
high_visibility_threshold = labeled['march_impressions'].quantile(0.75)

print(f"Stale threshold (known dates only): {stale_threshold:.0f} days")
print(f"Good position threshold: {good_position_threshold:.1f}")
print(f"High visibility threshold: {high_visibility_threshold:.0f} impressions")

Stale threshold (known dates only): 34 days
Good position threshold: 5.0
High visibility threshold: 1039 impressions


In [ ]:
def reason_code(row):
    good_position = row['march_avg_position'] <= good_position_threshold
    high_visibility = row['march_impressions'] >= high_visibility_threshold

    if row['declining'] == 1:
        if good_position and high_visibility:
            base = "Flagged declining despite ranking well and getting real traffic — check for an external cause (seasonality, SERP change, competitor)."
        elif good_position and not high_visibility:
            base = "Flagged declining with a good rank but low visibility — may just be losing an already-thin trickle of traffic."
        elif not good_position and high_visibility:
            base = "Flagged declining while still shown a lot — clicks appear to be drying up even as impressions hold."
        else:
            base = "Flagged declining with weak position and low visibility already — lowest-priority decline, may just be noise."
    else:
        if good_position and high_visibility:
            base = "Not flagged declining — ranks well and gets real traffic, currently stable."
        elif good_position and not high_visibility:
            base = "Not flagged declining — ranks well but barely seen, stable for now."
        elif not good_position and high_visibility:
            base = "Not flagged declining — shown a lot without ranking well, but holding steady."
        else:
            base = "Not flagged declining — low visibility and weak position, but no drop detected."

    if row['days_since_update'] != 9999:
        if row['days_since_update'] >= stale_threshold:
            base += f" Also hasn't been updated in {row['days_since_update']:.0f} days."
    else:
        base += " (Update history unknown.)"

    return base

labeled['reason_code'] = labeled.apply(reason_code, axis=1)
labeled[['content_hash_id', 'declining', 'march_avg_position', 'march_impressions', 'reason_code']].head(10)

,content_hash_id,declining,march_avg_position,march_impressions,reason_code
0,content_39d7361b4945d504,0,4.074107,77.0,Not flagged declining — ranks well but barely ...
1,content_c03ecafd4c999f15,0,8.240351,10849.0,Not flagged declining — shown a lot without ra...
2,content_e689bc511192751a,0,6.015432,61.0,Not flagged declining — low visibility and wea...
3,content_7dbc094b799e05a4,1,5.956862,705.0,Flagged declining with weak position and low v...
4,content_40b10da45f4c1cb5,0,12.977513,50.0,Not flagged declining — low visibility and wea...
5,content_df22bda1218f13ff,0,3.066796,2099.0,Not flagged declining — ranks well and gets re...
6,content_aa184c1b4ea518e5,0,9.444919,288.0,Not flagged declining — low visibility and wea...
7,content_b4de71c8ef5c4791,0,2.951764,3535.0,Not flagged declining — ranks well and gets re...
8,content_d7568011c4325a33,1,5.264774,1558.0,Flagged declining while still shown a lot — cl...
9,content_e847a4dcc8af3742,0,7.653508,2021.0,Not flagged declining — shown a lot without ra...


In [ ]:
test_df['declining_probability'] = model_scores
test_df['expected_clicks_at_risk'] = test_df['declining_probability'] * test_df['march_clicks']

# re-run your reason_code and action logic on test_df, since that's now your working table
test_df['reason_code'] = test_df.apply(reason_code, axis=1)

In [ ]:
def action_label(row):
    if row['declining'] != 1:
        return "No action — not flagged declining"

    good_position = row['march_avg_position'] <= good_position_threshold
    high_visibility = row['march_impressions'] >= high_visibility_threshold

    if good_position and high_visibility:
        return "Monitor"
    elif good_position and not high_visibility:
        return "Refresh"
    elif not good_position and high_visibility:
        return "Refresh"
    else:
        return "Deprioritize"

test_df['action'] = test_df.apply(action_label, axis=1)

In [ ]:
queue = test_df[test_df['declining'] == 1].sort_values('expected_clicks_at_risk', ascending=False)
queue = queue[['content_hash_id', 'client_hash_id', 'action', 'reason_code',
               'declining_probability', 'expected_clicks_at_risk', 'march_clicks']]

queue.head(15)

,content_hash_id,client_hash_id,action,reason_code,declining_probability,expected_clicks_at_risk,march_clicks
123510,content_512dbad65bd5ade9,client_73cda7b4e4f265ea,Monitor,Flagged declining despite ranking well and get...,1.000000,2506.000000,2506.0
25198,content_ec2e0346994fb5a5,client_e547b89c05043229,Monitor,Flagged declining despite ranking well and get...,1.000000,1480.000000,1480.0
38265,content_85703b835ab9e744,client_73cda7b4e4f265ea,Monitor,Flagged declining despite ranking well and get...,1.000000,1090.999968,1091.0
38368,content_987d251ee617d9c6,client_73cda7b4e4f265ea,Monitor,Flagged declining despite ranking well and get...,1.000000,939.999998,940.0
24622,content_18f0847d6628f8c6,client_e547b89c05043229,Monitor,Flagged declining despite ranking well and get...,0.999563,702.692983,703.0
34233,content_b2cb08ff59fcce78,client_73cda7b4e4f265ea,Monitor,Flagged declining despite ranking well and get...,0.999999,662.999150,663.0
121207,content_29c4a3831609805d,client_73cda7b4e4f265ea,Monitor,Flagged declining despite ranking well and get...,0.999834,637.894401,638.0
28756,content_7de2236ec61ba4c5,client_e547b89c05043229,Monitor,Flagged declining despite ranking well and get...,0.999815,632.882714,633.0
126833,content_57768353f230d65d,client_73cda7b4e4f265ea,Monitor,Flagged declining despite ranking well and get...,0.999999,580.999534,581.0
51843,content_ceb4692a9aa37b5e,client_73cda7b4e4f265ea,Refresh,Flagged declining while still shown a lot — cl...,0.992888,557.009896,561.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: This queue is meant for a content person to use as a starting point when deciding what to work on this week — not something that runs on its own. Every row is a suggestion, not an instruction. Someone should look at each one before acting on it.

Limits:

I only have honest predictions for 15 of the 47 clients — the ones the model was tested on, not trained on. The other 32 clients aren't covered by this queue at all.
This is built from one month's data (March into April). I don't know yet if the same patterns hold in other months or seasons.
Staleness data is missing for most pages (84%), so I only use it as an extra note, never as a main reason for an action.
For pages with a lot of clicks, the model's probability score maxes out near 1.0, so the ranking for those pages ends up mostly driven by click count, not real model confidence.
The model only looks at 4 numbers per page — it has no idea what the content is actually about, how well it's written, or what competitors are doing.
This is based on one train/test split, not cross-validated, so the exact numbers could shift a bit with a different split.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any row, I'd check:

Does this page belong to a client with an unusually large share of the queue? If so, check for a site-wide cause (tracking break, migration, algorithm hit) before treating each row as an independent page-level problem — this came directly from the client-concentration check in Section 1. Is declining_probability sitting at or very near 1.0 alongside a large march_clicks? Treat the ranking position as click-volume-driven, not as unusually high model confidence.
If a page's update history is unknown, I'd manually check it before deciding a refresh even makes sense.

Things that should never be automated:

Nothing gets published, edited, or removed by this pipeline on its own — every "Refresh" or "Deprioritize" is a suggestion a person has to look at first.
No client gets contacted or billed differently based on this queue without someone reviewing the actual pages first.
The model doesn't know anything about content quality, so a person always has to confirm a "Refresh" suggestion actually makes sense before any writing work starts.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

If a future month's precision score drops well below what I measured in w06 (0.500 at top-20, 0.380 at top-50), that means the model's not ranking as well anymore and the queue needs a re-check.
If the overall decline rate shifts a lot from what I'm seeing now (about 1 in 5 pages), that means something changed in the data and the model might not fit anymore.
If clients get added or removed, I'd need to rebuild the queue — right now it only knows about 15 of 47 clients.
If the missing-update-date problem gets fixed upstream (currently 84% missing), staleness should go back to being a real signal instead of just a footnote.
Since this is only built from one month's data, I'd treat it as outdated after about a quarter even if nothing else changes first.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os, json
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)
os.makedirs('work/metrics', exist_ok=True)

# 1. The ranked queue -- regenerated by the notebook, intentionally not committed (leak-guard)
queue.to_csv('work/outputs/w07_action_queue.csv', index=False)
print(f"Saved {len(queue)} rows to work/outputs/w07_action_queue.csv")

# 2. A reusable figure for the paper: action label distribution
action_counts = queue['action'].value_counts()
plt.figure(figsize=(6, 4))
action_counts.plot(kind='bar', color='#2B3A55')
plt.title('Action Queue: Recommended Action Distribution')
plt.ylabel('Number of pages')
plt.xlabel('Action')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('work/figures/w07_action_distribution.png', dpi=150)
plt.show()
print("Saved figure to work/figures/w07_action_distribution.png")

# 3. Metrics JSON -- the receipts the paper's numbers trace back to. Committed, unlike the raw queue.
metrics = {
    "test_clients": len(test_clients),
    "total_clients": len(unique_clients),
    "test_pages": len(test_df),
    "queue_pages": len(queue),
    "overall_test_base_rate": round(float(test_df['declining'].mean()), 3),
    "thresholds": {
        "stale_days": round(float(stale_threshold), 1),
        "good_position": round(float(good_position_threshold), 2),
        "high_visibility_impressions": round(float(high_visibility_threshold), 1),
    },
    "unknown_staleness_pages": int(unknown_staleness_pages),
    "unknown_staleness_share": round(float(unknown_staleness_pages / len(labeled)), 3),
    "action_distribution": action_counts.to_dict(),
    "known_limitations": [
        "Queue covers only the 15 held-out test clients, not all 47.",
        "declining_probability saturates near 1.0 for high-click pages -- priority score partially collapses to march_clicks at the top of the queue.",
        "Staleness unknown for 84% of pages; used as a secondary note only.",
        "Single client-grouped 70/30 split, one March-to-April snapshot -- not cross-validated, not tested across seasons."
    ]
}
with open('work/metrics/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Saved work/metrics/w07_playbook_metrics.json")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.